# Experiment Log Book

Evaluation results collected across experiments.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option("display.float_format", "{:.1f}".format)

RESULTS_ROOT = Path(
    "/work/dlclarge2/ferreira-oellm/open-instruct/oellm/evaluations/benchmarks/OpenJury/results"
)


def load_winrate(results_dir: str) -> pd.DataFrame:
    base = RESULTS_ROOT / results_dir
    rows = []
    for f in sorted(base.rglob("results-*.json")):
        d = json.loads(f.read_text())
        if d.get("eval_mode") != "winrate":
            continue
        model_b_wr = 1 - d["winrate"]
        rows.append({
            "Benchmark": d["dataset"],
            "Baseline WR%": d["winrate"] * 100,
            "Ours WR%": model_b_wr * 100,
            "Battles": d["num_battles"],
            "Wins": d["num_losses"],
            "Losses": d["num_wins"],
            "Ties": d["num_ties"],
        })
    if not rows:
        print(f"No winrate results found in {base}")
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    avg = df[["Baseline WR%", "Ours WR%"]].mean()
    avg_row = {"Benchmark": "Average", **avg.to_dict(), "Battles": df["Battles"].sum(),
               "Wins": df["Wins"].sum(), "Losses": df["Losses"].sum(), "Ties": df["Ties"].sum()}
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df


def load_rubric(results_dir: str) -> pd.DataFrame:
    base = RESULTS_ROOT / results_dir
    rows = []
    for f in sorted(base.rglob("results-*.json")):
        d = json.loads(f.read_text())
        if d.get("eval_mode") != "rubric":
            continue
        a_scores = d["model_A_scores"]
        b_scores = d["model_B_scores"]
        for criterion in d["criteria"]:
            key = f"{criterion}_score"
            rows.append({
                "Benchmark": d["dataset"],
                "Criterion": criterion.replace("_", " ").title(),
                "Baseline": a_scores[key],
                "Ours": b_scores[key],
                "Delta": b_scores[key] - a_scores[key],
            })
        rows.append({
            "Benchmark": d["dataset"],
            "Criterion": "Composite",
            "Baseline": a_scores["composite_score"] * 100,
            "Ours": b_scores["composite_score"] * 100,
            "Delta": (b_scores["composite_score"] - a_scores["composite_score"]) * 100,
        })
    if not rows:
        print(f"No rubric results found in {base}")
        return pd.DataFrame()
    return pd.DataFrame(rows)


def show_winrate(title, results_dir, ours_path, baseline_path, judge, date):
    df = load_winrate(results_dir)
    if df.empty:
        return
    display(Markdown(
        f"---\n## {title}\n\n"
        f"| | |\n|---|---|\n"
        f"| **Ours** | `{ours_path}` |\n"
        f"| **Baseline** | `{baseline_path}` |\n"
        f"| **Judge** | `{judge}` |\n"
        f"| **Date** | {date} |\n"
    ))
    display(df.style.format({
        "Baseline WR%": "{:.1f}", "Ours WR%": "{:.1f}",
        "Battles": "{:.0f}", "Wins": "{:.0f}", "Losses": "{:.0f}", "Ties": "{:.0f}",
    }).hide(axis="index").set_properties(
        subset=pd.IndexSlice[df.index[-1], :], **{"font-weight": "bold"}
    ))


def show_rubric(title, results_dir, ours_path, baseline_path, judge, date):
    df = load_rubric(results_dir)
    if df.empty:
        return
    display(Markdown(
        f"---\n## {title}\n\n"
        f"Per-criterion scores (1\u201310 scale). Composite is normalized 0\u2013100.\n\n"
        f"| | |\n|---|---|\n"
        f"| **Ours** | `{ours_path}` |\n"
        f"| **Baseline** | `{baseline_path}` |\n"
        f"| **Judge** | `{judge}` |\n"
        f"| **Date** | {date} |\n"
    ))
    _color = lambda v: "color: green" if isinstance(v, (int, float)) and v > 0 else (
        "color: red" if isinstance(v, (int, float)) and v < 0 else "")
    _styler = df.style.format({
        "Baseline": "{:.2f}", "Ours": "{:.2f}", "Delta": "{:+.2f}",
    }).hide(axis="index")
    _map_fn = getattr(_styler, "map", getattr(_styler, "applymap", None))
    _map_fn(_color, subset=["Delta"])
    display(_styler)

---
# Reproducing OLMo-3-7B-SFT

In [2]:
show_winrate(
    title="Think SFT v2 (HoreKa) \u2014 Winrate",
    results_dir="horeka-winrate-Olmo-3-7B-Think-SFT-20260224_141545",
    ours_path="checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf",
    baseline_path="models/baselines/Olmo-3-7B-Think-SFT",
    judge="Qwen/Qwen3-30B-A3B-Instruct-2507 (winrate, both orderings)",
    date="2026-02-24",
)

---
## Think SFT v2 (HoreKa) — Winrate

| | |
|---|---|
| **Ours** | `checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf` |
| **Baseline** | `models/baselines/Olmo-3-7B-Think-SFT` |
| **Judge** | `Qwen/Qwen3-30B-A3B-Instruct-2507 (winrate, both orderings)` |
| **Date** | 2026-02-24 |


Benchmark,Baseline WR%,Ours WR%,Battles,Wins,Losses,Ties
alpaca-eval,48.9,51.1,1610,819,783,8
arena-hard,49.5,50.5,1000,503,493,4
m-arena-hard-EU,48.8,51.2,12000,6129,5838,33
Average,49.1,50.9,14610,7451,7114,45


In [3]:
show_rubric(
    title="Think SFT v2 (HoreKa) \u2014 Rubric",
    results_dir="horeka-rubric-Olmo-3-7B-Think-SFT-20260224_161039",
    ours_path="checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf",
    baseline_path="models/baselines/Olmo-3-7B-Think-SFT",
    judge="Qwen/Qwen3-30B-A3B-Instruct-2507 (rubric)",
    date="2026-02-24",
)

---
## Think SFT v2 (HoreKa) — Rubric

Per-criterion scores (1–10 scale). Composite is normalized 0–100.

| | |
|---|---|
| **Ours** | `checkpoints/ferreira/olmo3-7b-sft/dolci-think-sft-v2-horeka-hf` |
| **Baseline** | `models/baselines/Olmo-3-7B-Think-SFT` |
| **Judge** | `Qwen/Qwen3-30B-A3B-Instruct-2507 (rubric)` |
| **Date** | 2026-02-24 |


Benchmark,Criterion,Baseline,Ours,Delta
alpaca-eval,Instruction Following,6.36,6.41,+0.05
alpaca-eval,Naturalness,6.81,6.77,-0.05
alpaca-eval,Coherence,6.70,6.66,-0.04
alpaca-eval,Accuracy,6.48,6.47,-0.01
alpaca-eval,Composite,93.13,92.95,-0.17
arena-hard,Instruction Following,5.80,5.82,+0.03
arena-hard,Naturalness,6.52,6.50,-0.02
arena-hard,Coherence,6.27,6.31,+0.03
arena-hard,Accuracy,5.89,5.95,+0.06
arena-hard,Composite,85.31,85.72,+0.41


---
*Add new experiments below by calling `show_winrate(...)` / `show_rubric(...)` in a new cell.*